# 第12回 画像生成 — 拡散モデルは、なぜノイズから絵を作れるのか

**このノートブックは第12回だけで完結します。** プログラミングの経験は要りません。

やること:
1. 画像にノイズを足して壊す(400歩で 1.7% まで消えます)
2. **拡散モデルを一から作って、自分で学習させる**
3. 完全なノイズから画像を作らせる
4. **生成物が訓練画像の写しかどうかを、測って確かめる**(著作権の論点)

所要時間:実習 約30分(うち学習に7〜12分)/ 課題 約40分

> **注意**:学習のセルは時間がかかります。実行を始めてから、待つあいだに説明を読んでください。
> 外部のモデルはダウンロードしません。すべてこのノートブックの中で作ります。

## 0. 準備

上から順にセルを実行してください(セルを選んで **Shift + Enter**)。

> **注意**:学習のセルに 7〜12分かかります。**先に実行を始めて、待つあいだに下を読んでください。**

In [ ]:
# 準備(このセルを最初に1回だけ実行)
!pip install -q japanize-matplotlib
import warnings; warnings.filterwarnings("ignore")
import torch, torch.nn as nn, numpy as np, time
import matplotlib.pyplot as plt, japanize_matplotlib
from torchvision import datasets, transforms
plt.rcParams["font.size"] = 12
torch.manual_seed(0)
print("準備できました")

In [ ]:
# 手書き数字を読み込む(初回は数十秒)
元データ = datasets.MNIST("./data", train=True, download=True, transform=transforms.ToTensor())
枚数 = 6000
画像 = torch.stack([元データ[i][0] for i in range(枚数)]) * 2 - 1   # −1〜1 にそろえる
print("学習に使う画像:", tuple(画像.shape))

fig, ax = plt.subplots(1, 8, figsize=(10, 1.5))
for i, a in enumerate(ax):
    a.imshow(画像[i, 0], cmap="gray", vmin=-1, vmax=1); a.axis("off")
plt.suptitle("これを学んで、似たものを作れるようにします")
plt.tight_layout(); plt.show()

## 1. きれいな画像を、ノイズで壊す

拡散モデルの出発点は、拍子抜けするほど単純です ── **画像にノイズを足して壊す**。

ここで大事なのは、**足したノイズの中身を、こちらが知っている**ことです(自分で作った乱数だから)。
つまり「ノイズ入りの画像」と「そこに足したノイズ」の組が、**人手をかけずに無限に作れます**。

第2回で「正解の札をつけるのが高い」という話をしました。拡散モデルは、その問題を回避しています。

In [ ]:
# ===== ① きれいな画像を、ノイズで壊す =====
T = 400                                    # ノイズを足す段階の数
beta = torch.linspace(1e-4, 0.04, T)       # 1歩ごとに足すノイズの量
alpha = 1 - beta
alphabar = torch.cumprod(alpha, 0)         # t歩後に「元画像が残る割合」の2乗

def ノイズを足す(画像, 歩, ノイズ=None):
    """t歩ぶんのノイズを一気に足す(1歩ずつやらなくても同じ結果になります)"""
    if ノイズ is None:
        ノイズ = torch.randn_like(画像)
    a = alphabar[歩]
    return a.sqrt() * 画像 + (1 - a).sqrt() * ノイズ, ノイズ

見本 = 画像[7:8]
torch.manual_seed(3)
共通ノイズ = torch.randn_like(見本)
段階 = [0, 25, 50, 100, 200, 300, 399]

fig, ax = plt.subplots(1, len(段階), figsize=(len(段階)*1.5, 1.9))
for a, t in zip(ax, 段階):
    x, _ = ノイズを足す(見本, t, 共通ノイズ)
    a.imshow(x[0, 0], cmap="gray", vmin=-1.5, vmax=1.5); a.axis("off")
    a.set_title(f"{t} 歩", fontsize=10)
plt.tight_layout(); plt.show()

print(f"{'歩数':>6}{'元画像との相関':>16}{'残る割合':>12}")
for t in 段階:
    x, _ = ノイズを足す(見本, t, 共通ノイズ)
    r = np.corrcoef(見本.flatten(), x.flatten())[0, 1]
    print(f"{t:>6}{r:>15.3f}{float(alphabar[t].sqrt())*100:>11.1f}%")
print("\n最初はゆっくり、あとで急に壊れます。100歩ではまだ元の形が見えるはずです。")

## 2. ノイズを当てるモデルを作る

> **なぜ「元の絵を当てよ」ではなく「足されたノイズを当てよ」なのか**
>
> 399歩ぶんのノイズがかかった画像には、元の絵が **1.7%** しか残っていません。
> そこから元の絵を復元するのは、砂嵐から写真を復元しろというのと同じで、無理です。
>
> ノイズのほうが単純です(ただの乱数の並び)。
> **難しい問題を、簡単な問題の積み重ねに分解した** ── これが拡散モデルの発想です。

In [ ]:
# ===== ② ノイズを当てるモデルを作る =====
# 「ノイズ入りの画像」と「何歩ぶんか」を入れると、
# 「足されたノイズ」を出力します。第6回の CNN によく似た形です。
class ノイズ予測(nn.Module):
    def __init__(self, c=48):
        super().__init__()
        self.t埋込 = nn.Sequential(nn.Linear(1, c), nn.SiLU(), nn.Linear(c, c))
        self.下1 = nn.Sequential(nn.Conv2d(1, c, 3, padding=1), nn.SiLU(),
                                 nn.Conv2d(c, c, 3, padding=1), nn.SiLU())
        self.下2 = nn.Sequential(nn.Conv2d(c, c*2, 3, stride=2, padding=1), nn.SiLU(),
                                 nn.Conv2d(c*2, c*2, 3, padding=1), nn.SiLU())
        self.中 = nn.Sequential(nn.Conv2d(c*2, c*2, 3, padding=1), nn.SiLU())
        self.上 = nn.Sequential(nn.ConvTranspose2d(c*2, c, 4, stride=2, padding=1), nn.SiLU())
        self.出 = nn.Sequential(nn.Conv2d(c*2, c, 3, padding=1), nn.SiLU(),
                                nn.Conv2d(c, 1, 3, padding=1))
    def forward(self, x, t):
        e = self.t埋込(t.float().unsqueeze(1) / T)        # 「何歩目か」も入力する
        h1 = self.下1(x) + e[:, :, None, None]
        h2 = self.中(self.下2(h1))
        return self.出(torch.cat([self.上(h2), h1], 1))

モデル = ノイズ予測()
print(f"重みの数: {sum(p.numel() for p in モデル.parameters()):,} 個")
print("(第10回でみた GPT-2 small は 1億2440万個。桁違いに小さいモデルです)")

In [ ]:
# 学習させる(7〜12分ほどかかります。実行したら次を読みながら待ってください)
最適化 = torch.optim.Adam(モデル.parameters(), lr=2e-3)
損失関数 = nn.MSELoss()
周回, まとめ = 14, 128
履歴 = []
開始 = time.time()

for 周 in range(周回):
    順番 = torch.randperm(枚数); 合計 = 0.0; 回数 = 0
    for i in range(0, 枚数, まとめ):
        xb = 画像[順番[i:i+まとめ]]
        歩 = torch.randint(0, T, (len(xb),))              # ランダムな段階を選ぶ
        a = alphabar[歩].view(-1, 1, 1, 1)
        ノイズ = torch.randn_like(xb)
        入力 = a.sqrt() * xb + (1 - a).sqrt() * ノイズ      # 問題を作る
        最適化.zero_grad()
        損失 = 損失関数(モデル(入力, 歩), ノイズ)           # 正解は「足したノイズ」
        損失.backward(); 最適化.step()
        合計 += float(損失); 回数 += 1
    履歴.append(合計 / 回数)
    print(f"  {周+1:>2}周目 loss {履歴[-1]:.4f}  ({time.time()-開始:.0f}秒)")

plt.figure(figsize=(6.5, 3.4))
plt.plot(range(1, 周回+1), 履歴, "o-", color="#3A46A8")
plt.xlabel("学習の周回"); plt.ylabel("loss"); plt.grid(alpha=.3)
plt.title("ノイズを当てる精度は上がったか")
plt.tight_layout(); plt.show()

## 3. 完全なノイズから、画像を作らせる

学習が終わったら、生成です。やることは1つ ──
**完全なノイズから始めて、「ノイズを当てる → 少し引く」を400回くり返す**だけです。

In [ ]:
# ===== ③ 完全なノイズから、画像を作らせる =====
@torch.no_grad()
def 生成(枚 , 歩数=T, seed=0):
    """ノイズから始めて、少しずつ引いていく"""
    g = torch.Generator().manual_seed(seed)
    x = torch.randn(枚, 1, 28, 28, generator=g)        # 完全なノイズ
    モデル.eval()
    for t in range(T-1, -1, -max(1, T // 歩数)):
        tt = torch.full((枚,), t, dtype=torch.long)
        予測ノイズ = モデル(x, tt)
        a, ab = alpha[t], alphabar[t]
        x = (x - (1-a) / (1-ab).sqrt() * 予測ノイズ) / a.sqrt()   # 少し引く
        if t > 0:
            x = x + beta[t].sqrt() * torch.randn(x.shape, generator=g)
    return x.clamp(-1, 1)

作った = 生成(16, T, seed=0)
fig, ax = plt.subplots(2, 8, figsize=(10, 2.6))
for i, a in enumerate(ax.flatten()):
    a.imshow(作った[i, 0], cmap="gray", vmin=-1, vmax=1); a.axis("off")
plt.suptitle("完全なノイズから作らせた16枚(400歩)")
plt.tight_layout(); plt.show()

print("手書きの線のようなものは出たでしょうか。")
print("はっきりした数字にはならないはずです ── 34万個の重み・十数分の学習では、ここが限界です。")
print("Stable Diffusion の初期版は重み約9億個。GPUを何百台も使って数週間学習しています。")

## 4. 何歩かけて戻すかで、何が変わるか

出発点(乱数の種)は同じにします。変えるのは「**何回に分けて引くか**」だけです。

In [ ]:
# ===== ④ 何歩かけて戻すかで、何が変わるか =====
# 出発点(乱数の種)は同じ。変えるのは「何回に分けて引くか」だけ。
平ら = 画像.reshape(枚数, -1); 平ら = 平ら / 平ら.norm(dim=1, keepdim=True)

歩たち = [5, 25, 100, 400]
fig, ax = plt.subplots(len(歩たち), 6, figsize=(8, len(歩たち)*1.4))
print(f"{'歩数':>6}{'訓練画像との最大類似度(平均)':>28}")
for r, 歩 in enumerate(歩たち):
    s = 生成(6, 歩, seed=0)
    F = s.reshape(6, -1); F = F / F.norm(dim=1, keepdim=True)
    類似 = float((F @ 平ら.T).max(dim=1).values.mean())
    print(f"{歩:>5}歩{類似:>25.3f}")
    for c in range(6):
        ax[r, c].imshow(s[c, 0], cmap="gray", vmin=-1, vmax=1); ax[r, c].axis("off")
    ax[r, 0].text(-0.35, 0.5, f"{歩} 歩", transform=ax[r, 0].transAxes,
                  fontsize=11, ha="right", va="center")
plt.tight_layout(); plt.show()

print("\n少しずつ引くほど、きれいになります。")
print("1回に大きく引くと、そのときの当て推量の誤りがそのまま残るからです。")
print("(第5回の学習率と同じ構造。ただし、そのぶん時間がかかります)")

## 5. 生成物は、訓練画像の写しなのか — 測る

著作権をめぐって、いま世界中で争われている論点です。
「学習に使った絵を、切り貼りしているだけではないか」。

**これは意見の問題に見えますが、測れます。**

ただし、**片方だけ測っても意味がありません。**
「生成画像と訓練画像が 85% 似ている」と言われても、それが高いのか低いのか判断できないからです。
だから比較対象として、**訓練画像どうしの類似度**も測ります。

In [ ]:
# ===== ⑤ 生成物は、訓練画像の写しなのか ── 測る =====
生成64 = 生成(64, T, seed=7)
F = 生成64.reshape(64, -1); F = F / F.norm(dim=1, keepdim=True)
類似行列 = F @ 平ら.T
最大類似 = 類似行列.max(dim=1).values
最近傍 = 類似行列.argmax(dim=1)

# 比較のため:訓練画像どうしの「いちばん近い他人」との類似度も測る
印 = torch.randperm(枚数)[:64]
S2 = 平ら[印] @ 平ら.T
S2[torch.arange(64), 印] = -1                 # 自分自身は除く
訓練最大 = S2.max(dim=1).values

print(f"生成画像 と いちばん近い訓練画像: 平均 {float(最大類似.mean()):.3f}  最大 {float(最大類似.max()):.3f}")
print(f"訓練画像 と 別の訓練画像(比較用): 平均 {float(訓練最大.mean()):.3f}  最大 {float(訓練最大.max()):.3f}")
print()
if 最大類似.mean() > 訓練最大.mean():
    print("→ 生成画像のほうが訓練画像に近い。写しの疑いがあります。")
else:
    print("→ 訓練画像どうしのほうが近い。この条件では、写しとは言えません。")

plt.figure(figsize=(7.5, 3.6))
plt.hist(最大類似.numpy(), bins=20, alpha=.7, label="生成 vs いちばん近い訓練画像", color="#E08A2B")
plt.hist(訓練最大.numpy(), bins=20, alpha=.7, label="訓練 vs 別の訓練画像", color="#3A46A8")
plt.xlabel("類似度"); plt.ylabel("枚数"); plt.legend(); plt.grid(alpha=.3)
plt.title("比べて初めて、数字の意味が決まります")
plt.tight_layout(); plt.show()

In [ ]:
# 目でも確かめる:類似度が最も高かった8組
順 = torch.argsort(-最大類似)[:8]
fig, ax = plt.subplots(2, 8, figsize=(10, 3))
for j, i in enumerate(順):
    ax[0, j].imshow(生成64[i, 0], cmap="gray", vmin=-1, vmax=1); ax[0, j].axis("off")
    ax[0, j].set_title(f"{最大類似[i]:.2f}", fontsize=9)
    ax[1, j].imshow(画像[最近傍[i], 0], cmap="gray", vmin=-1, vmax=1); ax[1, j].axis("off")
fig.text(0.02, 0.68, "生成", fontsize=11); fig.text(0.02, 0.25, "訓練", fontsize=11)
plt.suptitle("いちばん似ていた8組(上=生成 / 下=いちばん近い訓練画像)")
plt.tight_layout(); plt.show()

print("類似度の数字は高くても、目で見ると別物のはずです。")
print()
print("【ただし、ここで話を終えないでください】")
print("この実験で言えるのは「6000枚・重複なし・34万個のモデルでは写しは起きなかった」ことだけです。")
print("大きなモデルでは、学習データに重複して入っていた画像が、そのまま取り出せた報告があります")
print("(Carlini ら 2023)。条件が変われば、答えも変わります。")

> **今日いちばん持ち帰ってほしいこと**
>
> 「85% 似ている」は、写しの証拠にも、その反証にもなりません。
> 見るべきは**数字の大きさではなく、何と比べたか**です。
>
> 著作権の議論では、どちらの立場の人も数字を持ち出します。
> **比較のない数字は、印象を操作する道具になります。**

## 6. 自分で試す

**壊しても大丈夫**です(開き直せば元に戻ります)。

In [ ]:
# ===== 自分で試す(課題用)=====
# (a) ノイズの段階数を変えると、生成はどう変わるか
#     ※ T を変えたら、モデルを学習しなおす必要があります(時間がかかります)
#     ここでは「戻す歩数」だけを細かく変えてみましょう。
for 歩 in [3, 8, 15, 50, 200]:
    s = 生成(6, 歩, seed=1)
    F2 = s.reshape(6, -1); F2 = F2 / F2.norm(dim=1, keepdim=True)
    print(f"{歩:>4}歩 → 訓練画像との最大類似度 {float((F2 @ 平ら.T).max(dim=1).values.mean()):.3f}")

# (b) 乱数の種を変えると、違う絵が出るか
fig, ax = plt.subplots(3, 6, figsize=(8, 4))
for r, seed in enumerate([11, 22, 33]):
    s = 生成(6, T, seed=seed)
    for c in range(6):
        ax[r, c].imshow(s[c, 0], cmap="gray", vmin=-1, vmax=1); ax[r, c].axis("off")
    ax[r, 0].text(-0.3, 0.5, f"種 {seed}", transform=ax[r, 0].transAxes,
                  fontsize=10, ha="right", va="center")
plt.suptitle("出発点のノイズが違えば、出てくる絵も違う")
plt.tight_layout(); plt.show()
print("同じモデルでも、出発点のノイズが違えば違う絵になります。")
print("『毎回同じ絵しか作れない』わけではない、ということです。")

## 7. 提出課題(manaba)

**締切:次回講義の前日 23:59**

### 課題1:手で計算する(電卓可)
ある拡散モデルで、元画像の**残る割合**が次のようになりました。

```
0歩 100.0% / 50歩 95.0% / 100歩 82.0% / 200歩 35.0% / 300歩 9.0% / 399歩 2.0%
```

- (a) 0〜100歩 と 200〜300歩 で、減った量(ポイント)を比べなさい
- (b) この「最初はゆっくり、あとで急に」という性質は、**学習にとってなぜ都合が良い**のですか

### 課題2:ノートブックを動かして報告する
- (a) ノイズを足す段階の図と、loss の変化
- (b) 生成画像と、歩数(5/25/100/400)を変えた比較
- (c) **生成 vs 最近傍** と **訓練 vs 訓練** の類似度を、**必ず両方測って**比べる

> (c) で片方だけ測って結論している答案は評価しません。
> 比較対象がなければ、その数字が高いのか低いのか判断できないからです。

### 課題3:次の主張を批判する(300字以上)

> 「画像生成AI は学習データを写しているだけなので、著作権侵害です。
> 実際、生成画像は訓練画像と80%以上似ているという実験結果もあります。
> だから使用を全面的に禁止すべきです。」

**注意**:この主張を否定しろという課題ではありません。
結論に賛成でも反対でもかまいません。**論の運び方が正しいか**を検討してください。

考える材料:
- 「80%以上似ている」は、何と比べた数字ですか。今日の実測では何%でしたか
- 「写しているだけ」は、どんな条件で成り立ち、どんな条件で成り立ちませんか
- 「写し = 著作権侵害」は自明ですか。学習・出力・販売は同じ論点ですか
- 一部で問題が起きることから、「全面禁止」は導けますか

**加点**:「どういう条件なら問題で、どういう条件なら問題でないか」を切り分けられると評価が上がります。

---

採点は、正解かどうかよりも **試したことと考えたこと** を見ます。